In [1]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from googleapiclient.discovery import build
from google.oauth2 import service_account


periodos = ['26ES5', '2026D', '26V05']
periodo_virtual = '26V05'
periodo_presencial = '2026D'
periodo_especializacion = '26ES5'
periodo_pago = '202609'

In [2]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [3]:
novedades = pd.read_excel('NOVEDADES_REGIONALES_202609.xlsx')
novedades

,DOC_ALUM,Periodo,Programa,Regional,Asesor,Observaciones
0,1140902852,2026D,NEGOCIOS INTERNACIONALES,BARRANQUILLA,KATHERINE BOHORQUEZ,CUENTA CON MARCA DE PAGO Y ESTA OK EN ZOHO DEB...
1,1023864099,2026D,ADMINISTRACION DE EMPRESAS,BARRANQUILLA,KATHERINE BOHORQUEZ,YA ESTA EN ZOHO EN ESTADO MATRICULADA $120.000...
2,1236441592,2026D,ADMINISTRACION DE EMPRESAS,BARRANQUILLA,KATHERINE BOHORQUEZ,SE CANTO COMO ESPECIALIZACION Y ES PREGRADO PR...
3,1001820895,26V05,COMUNICACION SOCIAL,BARRANQUILLA,JESUS JIMENEZ,YA ESTA EN ZOHO EN ESTADO MATRICULADA Y FALTA ...
4,1143459132,26ES5,ESPECIALIZACION EN DESARROLLO ORGANIZACIONAL,BARRANQUILLA,JESUS JIMENEZ,YA ESTA EN ZOHO EN ESTADO MATRICULADA Y FALTA ...
...,...,...,...,...,...,...
147,1130277331,2026D,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,Medellin,Vinculaciones Laura Yovana Bedoya Ceballos,YA ESTÁ OK EN ZOHO
148,1007403332,26ES5,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,Tunja,Vinculaciones David Alejandro Orjuela Chaparro,PERTENECE A VINCULACIONES REGIONALES
149,71713801,26ES5,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,Medellin,Vinculaciones Laura Yovana Bedoya Ceballos,PERTENECE A VINCULACIONES REGIONALES
150,11106309,26ES5,ESPECIALIZACION EN CONTRATACION ESTATAL,Medellin,Vinculaciones David Quiceno Rendon,YA CUENTA CON MARCA DE PAGO


In [4]:
# Autenticación con la clave de servicio
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
KEY = 'key2.json'  # LLAVES
SPREADSHEET_ID = "15NkOzO6MBlMJjtabMTB1Bc_PEKJ6lb6Rh8won1i1xm0"  # ID DE LA HOJA DE CALCULO

# Crear credenciales
creds = service_account.Credentials.from_service_account_file(KEY, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)
sheet = service.spreadsheets()

# Llamar a la API de Google Sheets
RANGE_NAME = "'VENTAS'!A1:AE100000000"  # HOJA Y RANGO A TRABAJAR

result = sheet.values().get(spreadsheetId=SPREADSHEET_ID, range=RANGE_NAME).execute()
values = result.get('values', [])

# Mostrar resultados
if not values:
    print("No se encontraron datos.")
else:
    print(f"Datos obtenidos: {len(values):,} filas")

Datos obtenidos: 21,828 filas


In [5]:
df_excel = pd.DataFrame(values[1:], columns=values[0])
df_excel.shape

(21827, 31)

SE CRUZA CON EL REGISTRO DE VENTAS

In [6]:
df_excel['Número de Documento de estudiante'] = df_excel['Número de Documento de estudiante'].astype(str)
novedades['DOC_ALUM'] = novedades['DOC_ALUM'].astype(str)
df_pre_ventas = df_excel[df_excel['Número de Documento de estudiante'].isin(novedades['DOC_ALUM'])].copy()
df_pre_ventas.shape

(154, 31)

SE DEJAN LAS VENTAS QUE NO ESTAN EN EL REGISTRO DE VENTAS

In [7]:
df_no_preventas = df_pre_ventas[~df_pre_ventas['Número de Documento de estudiante'].isin(novedades['DOC_ALUM'])].copy()
df_no_preventas.shape

(0, 31)

# SE COMPLEMENTA LA INFORMACION DE LOS ASESORES

In [8]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [9]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    #archivo_orig = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.Com_Telec_Oct_2024_2_yCT", engine2_1)
    df_cedula = pd.read_sql(f'''
                    SELECT IDENTIFICACION AS 'Documento Ejecutivo', BOX_MAIL AS 'Correo Ejecutivo' FROM Hub_Empleados
                               ''', engine2_1)
    df_cedula_2 = df_cedula.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_cedula_2.shape

Conexion a la base de datos realizada
Conexión cerrada


(89782, 2)

In [10]:
df_cedula_2['Documento Ejecutivo'] = df_cedula_2['Documento Ejecutivo'].astype(int).astype(str)
df_cedula_2['Correo Ejecutivo'] = df_cedula_2['Correo Ejecutivo'].str.upper()

In [11]:
df_pre_ventas['Correo Ejecutivo'] = df_pre_ventas['Correo Ejecutivo'].str.upper()

In [12]:
# se unifica con la informacion de registro ventas
df_pre_ventas = df_pre_ventas.merge(df_cedula_2,how='left',left_on='Correo Ejecutivo',right_on='Correo Ejecutivo').copy()
df_pre_ventas.drop_duplicates(inplace=True)

SE TRAEN LAS METAS

In [13]:
df_metas_virtual = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_202609.xlsx', sheet_name= periodo_virtual)
df_metas_presencial = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_202609.xlsx', sheet_name= periodo_presencial)
df_metas_especializacion = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_202609.xlsx', sheet_name= periodo_especializacion)

In [14]:
df_metas_virtual['GESTOR CORREO'] = df_metas_virtual['GESTOR CORREO'].str.upper()
df_ventas_virtual = df_pre_ventas.copy()
df_ventas_virtual.shape

(154, 32)

In [15]:
# TRATAMIENTO DE COLUMNAS
columnas = ['Hora agregado', 'Numero de recibo','Dirección IP','Correo del estudiante','Teléfono del estudiante',
'Ciudad de residencia','Imagen del Pago Generado','% De Descuento',
'Medio de pago','Fuente de pionero','Campana','Fuente campana','Content','Sitio web', 'Tipo de inscripción',
'ID único','Fecha','Documento Ejecutivo_x','Número de aprobación de pago','Tipo de Financiación', 'Convenio', 'Valor pagado']

df_ventas_virtual.drop(columns=columnas, inplace=True)
df_ventas_virtual['Periodo'] = df_ventas_virtual['Periodo'].astype(str).str.strip()
df_ventas_virtual = df_ventas_virtual.rename(columns={'Documento Ejecutivo_y':'Documento Ejecutivo'})
df_ventas_virtual['Documento Ejecutivo'] = df_ventas_virtual['Documento Ejecutivo'].astype(str)
df_ventas_virtual['Número de Documento de estudiante'] = df_ventas_virtual['Número de Documento de estudiante'].astype(str)
df_ventas_virtual['Llave']=df_ventas_virtual['Número de Documento de estudiante']
df_ventas_virtual.shape

(154, 11)

# SE TRAE LA MARCA DE PAGO

In [16]:
periodos_sql = ",".join(f"'{p}'" for p in periodos)

In [17]:
para_orden = df_pre_ventas['Número de Documento de estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"

In [18]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
        SELECT distinct DOC_ALUM,PERIODO,ESTADO_PAGO,fuerzazoho as FuerzaComercialFinal, NUEVO, CLASE_ACTUAL, TIPO, PROGRAMA, EST_MATRICULADO
        FROM ZOHO.Base_Personas WITH(NOLOCK)
        WHERE PERIODO IN ({periodos_sql})
        and doc_alum in ({valores_unicos_df1_orden})
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_recibos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_recibos.shape

(137, 9)

In [19]:
df_recibos['LLAVE_RECIBOS'] = df_recibos['DOC_ALUM'].astype(str)
df_recibos_virtual = df_recibos.copy()
df_recibos_virtual.shape

(137, 10)

SE CRUZA LOS RECIBOS CON EL REGISTRO DE VENTAS

In [20]:
df_recibos_virtual = df_recibos_virtual.merge(df_ventas_virtual[['Propietario del registro','Modalidad','Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave']],how='inner',left_on='LLAVE_RECIBOS',right_on='Llave')
df_recibos_virtual.drop_duplicates(inplace=True)
df_recibos_virtual.shape

(142, 16)

In [21]:
#tratamiento de columnas
df_recibos_virtual['CLASE_ACTUAL'] = df_recibos_virtual['CLASE_ACTUAL'].astype(str).str.upper()
df_recibos_virtual['FuerzaComercialFinal'] = df_recibos_virtual['FuerzaComercialFinal'].str.upper()
df_recibos_virtual['NUEVO'] = df_recibos_virtual['NUEVO'].astype(str).str.upper()
df_recibos_virtual['ESTADO_PAGO'] = df_recibos_virtual['ESTADO_PAGO'].astype(str).str.upper()
df_recibos_virtual['FuerzaComercialFinal'] = df_recibos_virtual['FuerzaComercialFinal'].str.strip()
df_recibos_virtual['NUEVO'] = df_recibos_virtual['NUEVO'].str.strip()
df_recibos_virtual['CLASE_ACTUAL'] = df_recibos_virtual['CLASE_ACTUAL'].str.strip()
df_recibos_virtual['ESTADO_PAGO'] = df_recibos_virtual['ESTADO_PAGO'].str.strip()
df_recibos_virtual['EST_MATRICULADO'] = (
    df_recibos_virtual['EST_MATRICULADO']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [22]:
# se divide entre validos
df_validos_virtual = df_recibos_virtual[
    ((df_recibos_virtual['FuerzaComercialFinal'] == 'BASICA Y MEDIA')  | (df_recibos_virtual['FuerzaComercialFinal'] == 'VINCULACIONES') | (df_recibos_virtual['FuerzaComercialFinal'] == 'INDETERMINADO')) &
    (df_recibos_virtual['NUEVO'] == 'NUEVO') &
    (df_recibos_virtual['EST_MATRICULADO'] == '1-ACTIVO') & 
    (df_recibos_virtual['ESTADO_PAGO'] == 'PAGO')
].copy()
df_validos_virtual.drop_duplicates(inplace=True)
df_validos_virtual.shape

(123, 16)

In [23]:
# se divide entre no validos
mask = (
    df_recibos_virtual['FuerzaComercialFinal'].isin(
        ['BASICA Y MEDIA', 'VINCULACIONES', 'INDETERMINADO']
    ) &
    (df_recibos_virtual['NUEVO'] == 'NUEVO') &
    (df_recibos_virtual['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_recibos_virtual['ESTADO_PAGO'] == 'PAGO')
)
df_no_validos_virtual = df_recibos_virtual[~mask].copy()
df_no_validos_virtual.drop_duplicates(inplace=True)
df_no_validos_virtual.shape

(19, 16)

In [24]:
# Inicializar MOTIVO vacío
df_no_validos_virtual['MOTIVO'] = ''

# Condición 1
condicion = df_no_validos_virtual['ESTADO_PAGO'] == 'NO PAGO'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'NO REGISTRA PAGO | '

# Condición 2
condicion = (
    (df_no_validos_virtual['NUEVO'] != 'NUEVO')
)
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'NO ES UN ESTUDIANTE NUEVO | '

# Condición 3
condicion = df_no_validos_virtual['FuerzaComercialFinal'] != 'VINCULACIONES'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'PERTENECE A OTRA FUERZA | '

# Condición 4
condicion = df_no_validos_virtual['EST_MATRICULADO'] != '1-ACTIVO'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'MATRICULA INACTIVA | '

# Quitar el último " | "
df_no_validos_virtual['MOTIVO'] = (
    df_no_validos_virtual['MOTIVO']
    .str.rstrip(' |')
)

# Los que no tengan ningún motivo quedan como N/A
df_no_validos_virtual.loc[
    df_no_validos_virtual['MOTIVO'] == '',
    'MOTIVO'
] = 'N/A'

df_no_validos_virtual

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
7,1002600121,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,INGENIERÍA DE SISTEMAS,1-ACTIVO,1002600121,Vinculaciones Miryam Daniela Callejas Perez,Virtual,MIRYAM_CALLEJASP@CUN.EDU.CO,Tunja,1053616791,1002600121,NO REGISTRA PAGO
18,1006510343,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,1006510343,Vinculaciones Luis Alejandro Muñoz,Virtual,LUIS_MUNOZMU@CUN.EDU.CO,Pereira,1023938622,1006510343,NO REGISTRA PAGO
19,1007403332,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,1007403332,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,1007403332,PERTENECE A OTRA FUERZA
50,1057411128,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,1057411128,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,1057411128,NO REGISTRA PAGO
71,1069727030,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DEL...,1-ACTIVO,1069727030,Vinculaciones Luis Alejandro Muñoz,Virtual,LUIS_MUNOZMU@CUN.EDU.CO,Pereira,1023938622,1069727030,NO REGISTRA PAGO
73,1072098738,2026D,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,1072098738,Vinculaciones Nicolas Viafara Ospina,Presencial,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,1072098738,PERTENECE A OTRA FUERZA
76,1072432439,26V05,NO PAGO,VINCULACIONES,NUEVO,REINTEGRO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS CONTABLES,1-ACTIVO,1072432439,Vinculaciones Jesus David Jimenez Amado,Virtual,JESUS_JIMENEZ@CUN.EDU.CO,Barranquilla,1081811110,1072432439,NO REGISTRA PAGO
80,1081026220,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,1081026220,Vinculaciones Jesus David Jimenez Amado,Virtual,JESUS_JIMENEZ@CUN.EDU.CO,Barranquilla,1081811110,1081026220,PERTENECE A OTRA FUERZA
83,1083876368,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,1-ACTIVO,1083876368,Vinculaciones Jose Andres Navarro Ramirez,Virtual,JOSE_NAVARRORA@CUN.EDU.CO,Neiva,1075230146,1083876368,NO REGISTRA PAGO
85,1094926264,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS CONTABLES,1-ACTIVO,1094926264,Vinculaciones Alejandro Tabares Buitrago,Virtual,ALEJANDRO_TABARES@CUN.EDU.CO,Pereira,18523096,1094926264,PERTENECE A OTRA FUERZA


In [25]:
# se traen las ventas que no tienen marca de pago
df_no_recibos_virtual = df_ventas_virtual[~df_ventas_virtual['Llave'].isin(df_recibos_virtual['LLAVE_RECIBOS'])].copy()
df_no_recibos_virtual['DOC_ALUM'] = df_no_recibos_virtual['Número de Documento de estudiante'].astype(str)
df_no_recibos_virtual['PERIODO'] = df_no_recibos_virtual['Periodo']
df_no_recibos_virtual['ESTADO_PAGO'] = ' '
df_no_recibos_virtual['EST_MATRICULADO'] = ' '
df_no_recibos_virtual['FuerzaComercialFinal'] = ' '
df_no_recibos_virtual['NUEVO'] = ' '
df_no_recibos_virtual['CLASE_ACTUAL'] = ' '
df_no_recibos_virtual['TIPO'] = ' '
df_no_recibos_virtual['PROGRAMA'] = df_no_recibos_virtual['Programa vendido'] 
df_no_recibos_virtual['LLAVE_RECIBOS'] = ' '
df_no_recibos_virtual['MOTIVO'] = 'NO CUENTA CON MARCA DE PAGO PARA ESE PERIODO'
df_no_recibos_virtual = df_no_recibos_virtual[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO', 'LLAVE_RECIBOS',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave','MOTIVO']]
df_no_recibos_virtual.shape

(15, 17)

In [26]:
# SE UNIFICAN LOS NO VALIDOS Y LOS QUE NO TIENEN MARCA DE PAGO
df_no_validos_2 = pd.concat([df_no_validos_virtual,df_no_recibos_virtual])
df_no_validos_2.shape

(34, 17)

SE TRAEN LAS OPORTUNIDADES DE ZOHO CRM

In [27]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_zoho = pd.read_sql(f'''
Select distinct 
o.Propietario_de_Oportunidad_Name,
o.Hora_de_creación,
o.Fecha_de_cierre,
o.Fase,
o.Hora_de_modificación,
o.Número_de_Documento,
o.Programa_de_interés,
o.PERIODO,
o.Hora_de_la_última_actividad,
u.Correo_electrónico
from zoho.CRM.Oportunidades o
left join zoho.CRM.Usuarios U
ON o.Propietario_de_Oportunidad = u.Id
    where Periodo in ({periodos_sql})
and Fase LIKE '%Matriculado%'
and o.Workdrive_Folder_ID is not null
                               ''', engine2_1)
    Consulta_z2 = df_zoho.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_zoho.shape

Conexion a la base de datos realizada
Conexión cerrada


(14977, 10)

In [28]:
df_zoho['Correo_electrónico'] = df_zoho['Correo_electrónico'].str.upper()
df_zoho['LLAVE_ZOHO']=df_zoho['Número_de_Documento'].astype(str)+df_zoho['PERIODO'].astype(str).str.strip()+df_zoho['Correo_electrónico']
df_zoho_vinculaciones = df_zoho[df_zoho['Correo_electrónico'].isin(df_pre_ventas['Correo Ejecutivo'])].copy()
df_zoho_vinculaciones = df_zoho_vinculaciones.drop_duplicates()
df_zoho_vinculaciones.shape

(1269, 11)

In [29]:
# se crea la llave para cruzar por doc alum+periodo+correo ejecutivo
df_validos_virtual['LLAVE_ZOHO'] = df_validos_virtual['DOC_ALUM']+df_validos_virtual['PERIODO']+df_validos_virtual['Correo Ejecutivo']

In [30]:
df_zoho_recibos_virtual = df_validos_virtual.merge(df_zoho_vinculaciones[['Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación','LLAVE_ZOHO']],left_on='LLAVE_ZOHO',right_on='LLAVE_ZOHO',how='inner').copy()
df_zoho_recibos_virtual.drop_duplicates(inplace=True)
df_zoho_recibos_virtual.shape

(100, 20)

In [31]:
df_no_validos_zoho = df_validos_virtual[~df_validos_virtual['LLAVE_ZOHO'].isin(df_zoho_vinculaciones['LLAVE_ZOHO'])].copy()
df_no_validos_zoho['MOTIVO'] = 'NO ESTA OK EN ZOHO'
df_no_validos_zoho = df_no_validos_zoho.drop(columns=['LLAVE_ZOHO'])
df_no_validos_zoho.shape

(23, 17)

In [32]:
# SE UNIFICAN LOS NO VALIDOS TOTALES DE VIRTUALES
df_no_validos_3 = pd.concat([df_no_validos_2,df_no_validos_zoho])
df_no_validos_3.shape

(57, 17)

# SE VALIDA QUE NO SE HAYA PAGADO 

In [33]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 
            select * from CUN_REPOSITORIO.AARBOLEDA.COMISIONES_VINCULACIONES_HIST
            ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_historico.shape

(1357, 18)

In [34]:
df_historico['LLAVE_HISTORICO'] = df_historico['DOC_ALUM'].astype(str) + df_historico['PERIODO'].astype(str)

In [35]:
#------------------------------PROVISIONAL ASI DEBERIA SER LA  LLAVE DE T00000D0 EL PROCESO-----------------------
df_zoho_recibos_virtual['Llave'] = df_zoho_recibos_virtual['DOC_ALUM'].astype(str) + df_zoho_recibos_virtual['PERIODO'].astype(str)

In [36]:
df_validos_virtuales_final = df_zoho_recibos_virtual[~df_zoho_recibos_virtual['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_validos_virtuales_final.shape

(44, 20)

In [37]:
df_no_validos_historico = df_zoho_recibos_virtual[df_zoho_recibos_virtual['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_no_validos_historico['MOTIVO'] = 'YA SE CONTO EN LA LIQUIDACION' 
df_no_validos_historico = df_no_validos_historico.drop(columns=['LLAVE_ZOHO','Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación'])
df_no_validos_historico.shape

(56, 17)

In [38]:
df_novalidos_virtual_final = pd.concat([df_no_validos_3,df_no_validos_historico])
df_novalidos_virtual_final.shape

(113, 17)

# SE TRAE EL HISTORICO DE VENTAS PARA REALIZAR LOS CALCULOS

In [39]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
            select * from CUN_REPOSITORIO.AARBOLEDA.BASEPAGO_VINCULACIONES_HIST
                         WHERE PERIODO_PAGO = ({periodo_pago})
            ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico_basepago = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_historico_basepago.shape

(41, 24)

In [40]:
pre_ventas_asesor_virtuales = (
    df_validos_virtuales_final[df_validos_virtuales_final['PERIODO'] == periodo_virtual]
    .groupby('Documento Ejecutivo')
    .size()
    .reset_index(name='PRE_VENTAS_VIRTUALES')
)
pre_ventas_asesor_virtuales.shape

(11, 2)

In [41]:
pre_ventas_asesor_virtuales['Documento Ejecutivo'] = pre_ventas_asesor_virtuales['Documento Ejecutivo'].astype(str)
df_historico_basepago['Documento Ejecutivo'] = df_historico_basepago['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_virtuales = pre_ventas_asesor_virtuales.merge(df_historico_basepago[['Documento Ejecutivo','VENTAS_VIRTUALES']],how='right',on='Documento Ejecutivo')
pre_ventas_asesor_virtuales['PRE_VENTAS_VIRTUALES'] = (
    pre_ventas_asesor_virtuales['PRE_VENTAS_VIRTUALES'].fillna(0)
)

pre_ventas_asesor_virtuales['VENTAS_VIRTUALES'] = (
    pre_ventas_asesor_virtuales['VENTAS_VIRTUALES'].fillna(0)
)

pre_ventas_asesor_virtuales['TOTAL_VENTAS'] = (
    pre_ventas_asesor_virtuales['PRE_VENTAS_VIRTUALES'] +
    pre_ventas_asesor_virtuales['VENTAS_VIRTUALES']
)


In [42]:
pre_ventas_asesor_virtuales['Documento Ejecutivo'] = pre_ventas_asesor_virtuales['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_virtuales = pre_ventas_asesor_virtuales.drop(columns=['PRE_VENTAS_VIRTUALES','VENTAS_VIRTUALES'])
pre_ventas_asesor_virtuales = pre_ventas_asesor_virtuales.rename(columns={'TOTAL_VENTAS':'VENTAS_VIRTUALES'})

In [43]:
df_metas_virtual['GESTOR DOCUMENTO'] = df_metas_virtual['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_virtuales = pre_ventas_asesor_virtuales.merge(df_metas_virtual,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_virtuales = ventas_asesor_virtuales.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_virtuales

,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1000224303,2.0,GUISELLE_VALENZUELA@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE
1,1001344725,4.0,JULIANA_AUDOR@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA
2,1005337691,7.0,KAREN_AGUILAR@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,BUCARAMANGA,BUCARAMANGA
3,1006053851,1.0,VALENTINA_MONTOYAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,CALI,CALI
4,1023938622,6.0,LUIS_MUNOZMU@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,PEREIRA,PEREIRA
5,1025521082,4.0,DAVID_ORJUELAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA
6,1033377195,2.0,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN
7,1036672351,20.0,JUAN_BRANPE@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN
8,1053616791,3.0,MIRYAM_CALLEJASP@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA
9,1053787428,18.0,DAVID_QUICENO@CUN.EDU.CO,Gestor de ventas Senior,16.0,SANDRA,MEDELLIN,MEDELLIN


# SE DEFINE EL ESQUEMA COMISIONAL

In [44]:
esquema_comisiones_virtuales = {

    "SANDRA": {
        "BARRANQUILLA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},

        "CARTAGENA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 5, 23000),
                                (5, 7, 100000),
                                (8, float('inf'), 130000)
                            
                            ]       
            }},
        "IBAGUE": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 30, 23000),
                                (30, 37, 90000),
                                (38, float('inf'), 120000)
                            
                            ]       
            }},
        "MEDELLIN": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},
        "MONTERIA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},
        "NEIVA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 5, 23000),
                                (5, 7, 90000),
                                (8, float('inf'), 120000)
                            
                            ]       
            }},
        "SANTA MARTA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},
        "SINCELEJO": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},               
        "CALI": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]            
            }}, 
        "PEREIRA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]     
                            }},
        "BUCARAMANGA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]     }},
         "TUNJA": {
            "Gestor de ventas Senior":{ 
                        17: [  
                                (17, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]  },
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]       
            }}            
    }}

In [45]:
def asignar_comision_virtuales(lider, sede, cargo, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_virtuales:
        return 0

    # 2️⃣ Validar sede
    if sede not in esquema_comisiones_virtuales[lider]:
        return 0

    # 3️⃣ Validar cargo
    if cargo not in esquema_comisiones_virtuales[lider][sede]:
        return 0

    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_virtuales[lider][sede][cargo]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_virtuales[lider][sede][cargo][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [46]:
ventas_asesor_virtuales["COMISION_TOTAL_VIRTUAL"] = ventas_asesor_virtuales.apply(
    lambda row: asignar_comision_virtuales(
        row["LIDER"],
        row["SEDE"],
        row["CARGO"],
        row["Meta"],
        row["VENTAS_VIRTUALES"]
    ),
    axis=1
)
ventas_asesor_virtuales['COMISION_UNITARIA_VIRTUAL'] = ventas_asesor_virtuales['COMISION_TOTAL_VIRTUAL'] / ventas_asesor_virtuales['VENTAS_VIRTUALES']
ventas_asesor_virtuales['%CUMPLIMIENTO_VIRTUAL'] = (
    (ventas_asesor_virtuales['VENTAS_VIRTUALES'] / ventas_asesor_virtuales['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_virtuales

,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_VIRTUAL,COMISION_UNITARIA_VIRTUAL,%CUMPLIMIENTO_VIRTUAL
0,1000224303,2.0,GUISELLE_VALENZUELA@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE,46000.0,23000.0,200.0%
1,1001344725,4.0,JULIANA_AUDOR@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA,92000.0,23000.0,400.0%
2,1005337691,7.0,KAREN_AGUILAR@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,BUCARAMANGA,BUCARAMANGA,0.0,0.0,33.33%
3,1006053851,1.0,VALENTINA_MONTOYAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,CALI,CALI,0.0,0.0,4.76%
4,1023938622,6.0,LUIS_MUNOZMU@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,PEREIRA,PEREIRA,0.0,0.0,28.57%
5,1025521082,4.0,DAVID_ORJUELAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA,0.0,0.0,19.05%
6,1033377195,2.0,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN,0.0,0.0,11.76%
7,1036672351,20.0,JUAN_BRANPE@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN,1800000.0,90000.0,117.65%
8,1053616791,3.0,MIRYAM_CALLEJASP@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA,0.0,0.0,14.29%
9,1053787428,18.0,DAVID_QUICENO@CUN.EDU.CO,Gestor de ventas Senior,16.0,SANDRA,MEDELLIN,MEDELLIN,1620000.0,90000.0,112.5%


# PROCESO DE COMISION PRESENCIALES

In [47]:
pre_ventas_asesor_presenciales = (
    df_validos_virtuales_final[df_validos_virtuales_final['PERIODO'] == periodo_presencial]
    .groupby('Documento Ejecutivo')
    .size()
    .reset_index(name='PRE_VENTAS_PRESENCIALES')
)
pre_ventas_asesor_presenciales.shape

(5, 2)

In [48]:
pre_ventas_asesor_presenciales['Documento Ejecutivo'] = pre_ventas_asesor_presenciales['Documento Ejecutivo'].astype(str)
df_historico_basepago['Documento Ejecutivo'] = df_historico_basepago['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_presenciales = pre_ventas_asesor_presenciales.merge(df_historico_basepago[['Documento Ejecutivo','VENTAS_PRESENCIALES']],how='right',on='Documento Ejecutivo')
pre_ventas_asesor_presenciales['VENTAS_PRESENCIALES'] = (pre_ventas_asesor_presenciales['VENTAS_PRESENCIALES'].fillna(0))
pre_ventas_asesor_presenciales['PRE_VENTAS_PRESENCIALES'] = (pre_ventas_asesor_presenciales['PRE_VENTAS_PRESENCIALES'].fillna(0))
pre_ventas_asesor_presenciales['TOTAL_VENTAS'] = (pre_ventas_asesor_presenciales['PRE_VENTAS_PRESENCIALES'] + pre_ventas_asesor_presenciales['VENTAS_PRESENCIALES'])
pre_ventas_asesor_presenciales.shape

(41, 4)

In [49]:
pre_ventas_asesor_presenciales['Documento Ejecutivo'] = pre_ventas_asesor_presenciales['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_presenciales = pre_ventas_asesor_presenciales.drop(columns=['PRE_VENTAS_PRESENCIALES','VENTAS_PRESENCIALES'])
pre_ventas_asesor_presenciales = pre_ventas_asesor_presenciales.rename(columns={'TOTAL_VENTAS':'VENTAS_PRESENCIALES'})

In [50]:
df_metas_presencial['GESTOR DOCUMENTO'] = df_metas_presencial['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_presenciales = pre_ventas_asesor_presenciales.merge(df_metas_presencial,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_presenciales = ventas_asesor_presenciales.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_presenciales

,Documento Ejecutivo,VENTAS_PRESENCIALES,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1000224303,0.0,guiselle_valenzuela@cun.edu.co,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE
1,1001344725,0.0,juliana_audor@cun.edu.co,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA
2,1005337691,0.0,karen_aguilar@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,BUCARAMANGA,BUCARAMANGA
3,1006053851,0.0,valentina_montoyac@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,CALI,CALI
4,1023938622,0.0,luis_munozmu@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,PEREIRA,PEREIRA
5,1025521082,0.0,david_orjuelac@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,TUNJA,TUNJA
6,1033377195,2.0,laura_bedoyac@cun.edu.co,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN
7,1036672351,0.0,juan_branpe@cun.edu.co,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN
8,1053616791,0.0,miryam_callejasp@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,TUNJA,TUNJA
9,1053787428,22.0,david_quiceno@cun.edu.co,Gestor de ventas Senior,16.0,SANDRA,MEDELLIN,MEDELLIN


In [51]:
esquema_comisiones_presenciales = {

    "SANDRA": {
        "BARRANQUILLA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 34, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},

        "CARTAGENA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 5, 23000),
                                (5, 7, 100000),
                                (8, float('inf'), 130000)
                            
                            ]       
            }},
        "IBAGUE": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 18, 23000),
                                (18, 35, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},
        "MEDELLIN": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},
        "MONTERIA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},
        "NEIVA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 3, 23000),
                                (3, 5, 90000),
                                (6, float('inf'), 120000)
                            
                            ]       
            }},
        "SANTA MARTA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},
        "SINCELEJO": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},               
        "CALI": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]            
            }}, 
        "PEREIRA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]     
                            }},
        "BUCARAMANGA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]     }},
         "TUNJA": {
            "Gestor de ventas Senior":{ 
                        17: [  
                                (17, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]  },
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]       
            }}            
    }}

In [52]:
def asignar_comision_presenciales(lider, sede, cargo, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_presenciales:
        return 0

    # 2️⃣ Validar sede
    if sede not in esquema_comisiones_presenciales[lider]:
        return 0

    # 3️⃣ Validar cargo
    if cargo not in esquema_comisiones_presenciales[lider][sede]:
        return 0

    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_presenciales[lider][sede][cargo]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_presenciales[lider][sede][cargo][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [53]:
ventas_asesor_presenciales["COMISION_TOTAL_PRESENCIAL"] = ventas_asesor_presenciales.apply(
    lambda row: asignar_comision_presenciales(
        row["LIDER"],
        row["SEDE"],
        row["CARGO"],
        row["Meta"],
        row["VENTAS_PRESENCIALES"]
    ),
    axis=1
)
ventas_asesor_presenciales['COMISION_UNITARIA_PRESENCIAL'] = ventas_asesor_presenciales['COMISION_TOTAL_PRESENCIAL'] / ventas_asesor_presenciales['VENTAS_PRESENCIALES']
ventas_asesor_presenciales['%CUMPLIMIENTO_PRESENCIAL'] = (
    (ventas_asesor_presenciales['VENTAS_PRESENCIALES'] / ventas_asesor_presenciales['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_presenciales

,Documento Ejecutivo,VENTAS_PRESENCIALES,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_PRESENCIAL,COMISION_UNITARIA_PRESENCIAL,%CUMPLIMIENTO_PRESENCIAL
0,1000224303,0.0,guiselle_valenzuela@cun.edu.co,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE,0.0,NaN,0.0%
1,1001344725,0.0,juliana_audor@cun.edu.co,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA,0.0,NaN,0.0%
2,1005337691,0.0,karen_aguilar@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,BUCARAMANGA,BUCARAMANGA,0.0,NaN,nan%
3,1006053851,0.0,valentina_montoyac@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,CALI,CALI,0.0,NaN,nan%
4,1023938622,0.0,luis_munozmu@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,PEREIRA,PEREIRA,0.0,NaN,nan%
5,1025521082,0.0,david_orjuelac@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,TUNJA,TUNJA,0.0,NaN,nan%
6,1033377195,2.0,laura_bedoyac@cun.edu.co,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN,0.0,0.0,11.76%
7,1036672351,0.0,juan_branpe@cun.edu.co,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN,0.0,NaN,0.0%
8,1053616791,0.0,miryam_callejasp@cun.edu.co,Gestor de ventas Junior,0.0,SANDRA,TUNJA,TUNJA,0.0,NaN,nan%
9,1053787428,22.0,david_quiceno@cun.edu.co,Gestor de ventas Senior,16.0,SANDRA,MEDELLIN,MEDELLIN,1980000.0,90000.0,137.5%


#PROCESO DE LIQUIDACION DE ESPECIALIZACIONES

In [54]:
pre_ventas_asesor_especializaciones = (
    df_validos_virtuales_final[df_validos_virtuales_final['PERIODO'] == periodo_especializacion]
    .groupby('Documento Ejecutivo')
    .size()
    .reset_index(name='PRE_VENTAS_ESPECIALIZACION')
)
pre_ventas_asesor_especializaciones.shape

(9, 2)

In [55]:
pre_ventas_asesor_especializaciones['Documento Ejecutivo'] = pre_ventas_asesor_especializaciones['Documento Ejecutivo'].astype(str)
df_historico_basepago['Documento Ejecutivo'] = df_historico_basepago['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_especializaciones = pre_ventas_asesor_especializaciones.merge(df_historico_basepago[['Documento Ejecutivo','VENTAS_ESPECIALIZACION']],how='right',on='Documento Ejecutivo')
pre_ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'] = (pre_ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'].fillna(0))
pre_ventas_asesor_especializaciones['PRE_VENTAS_ESPECIALIZACION'] = (pre_ventas_asesor_especializaciones['PRE_VENTAS_ESPECIALIZACION'].fillna(0))
pre_ventas_asesor_especializaciones['TOTAL_VENTAS'] = (pre_ventas_asesor_especializaciones['PRE_VENTAS_ESPECIALIZACION'] + pre_ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'])
pre_ventas_asesor_especializaciones.shape

(41, 4)

In [56]:
pre_ventas_asesor_especializaciones['Documento Ejecutivo'] = pre_ventas_asesor_especializaciones['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_especializaciones = pre_ventas_asesor_especializaciones.drop(columns=['PRE_VENTAS_ESPECIALIZACION','VENTAS_ESPECIALIZACION'])
pre_ventas_asesor_especializaciones = pre_ventas_asesor_especializaciones.rename(columns={'TOTAL_VENTAS':'VENTAS_ESPECIALIZACION'})

In [57]:
df_metas_especializacion['GESTOR DOCUMENTO'] = df_metas_especializacion['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_especializaciones = pre_ventas_asesor_especializaciones.merge(df_metas_especializacion,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_especializaciones = ventas_asesor_especializaciones.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_especializaciones

,Documento Ejecutivo,VENTAS_ESPECIALIZACION,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1000224303,1.0,guiselle_valenzuela@cun.edu.co,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE
1,1001344725,0.0,juliana_audor@cun.edu.co,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA
2,1005337691,2.0,karen_aguilar@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,BUCARAMANGA,BUCARAMANGA
3,1006053851,23.0,valentina_montoyac@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,CALI,CALI
4,1023938622,1.0,luis_munozmu@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,PEREIRA,PEREIRA
5,1025521082,19.0,david_orjuelac@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,TUNJA,TUNJA
6,1033377195,27.0,laura_bedoyac@cun.edu.co,Gestor de ventas Junior,12.0,SANDRA,MEDELLIN,MEDELLIN
7,1036672351,0.0,juan_branpe@cun.edu.co,Gestor de ventas Junior,12.0,SANDRA,MEDELLIN,MEDELLIN
8,1053616791,23.0,miryam_callejasp@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,TUNJA,TUNJA
9,1053787428,1.0,david_quiceno@cun.edu.co,Gestor de ventas Senior,8.0,SANDRA,MEDELLIN,MEDELLIN


In [58]:
esquema_comisiones_especializaciones = {

    "SANDRA": {
        "BARRANQUILLA": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},

        "CARTAGENA": {
            "Gestor de ventas Senior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 5, 23000),
                                (5, 7, 100000),
                                (8, float('inf'), 130000)
                            
                            ]       
            }},
        "IBAGUE": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 12, 23000),
                                (13, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},
        "MEDELLIN": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},
        "MONTERIA": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},
        "NEIVA": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 3, 23000),
                                (3, 5, 90000),
                                (6, float('inf'), 120000)
                            
                            ]       
            }},
        "SANTA MARTA": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},
        "SINCELEJO": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},               
        "CALI": {
            "Gestor de ventas Senior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        15: [  
                                (15, 19, 100000),
                                (20, float('inf'), 130000)
                            ]          
            }}, 
        "PEREIRA": {
            "Gestor de ventas Senior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        15: [  
                                (15, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ],
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]     
                            }},
        "BUCARAMANGA": {
            "Gestor de ventas Senior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ],
                        15: [  
                                (15, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]     }},
         "TUNJA": {
            "Gestor de ventas Senior":{ 
                        11: [  
                                (11, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]  },
            "Gestor de ventas Junior":{ 
                        15: [  
                                (15, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]       
            }}            
    }}

In [59]:
def asignar_comision_especializaciones(lider, sede, cargo, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_especializaciones:
        return 0

    # 2️⃣ Validar sede
    if sede not in esquema_comisiones_especializaciones[lider]:
        return 0

    # 3️⃣ Validar cargo
    if cargo not in esquema_comisiones_especializaciones[lider][sede]:
        return 0

    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_especializaciones[lider][sede][cargo]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_especializaciones[lider][sede][cargo][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [60]:
ventas_asesor_especializaciones["COMISION_TOTAL_ESPECIALIZACION"] = ventas_asesor_especializaciones.apply(
    lambda row: asignar_comision_especializaciones(
        row["LIDER"],
        row["SEDE"],
        row["CARGO"],
        row["Meta"],
        row["VENTAS_ESPECIALIZACION"]
    ),
    axis=1
)
ventas_asesor_especializaciones['COMISION_UNITARIA_ESPECIALIZACION'] = ventas_asesor_especializaciones['COMISION_TOTAL_ESPECIALIZACION'] / ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION']
ventas_asesor_especializaciones['%CUMPLIMIENTO_ESPECIALIZACION'] = (
    (ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'] / ventas_asesor_especializaciones['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_especializaciones

,Documento Ejecutivo,VENTAS_ESPECIALIZACION,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_ESPECIALIZACION,COMISION_UNITARIA_ESPECIALIZACION,%CUMPLIMIENTO_ESPECIALIZACION
0,1000224303,1.0,guiselle_valenzuela@cun.edu.co,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE,23000.0,23000.0,100.0%
1,1001344725,0.0,juliana_audor@cun.edu.co,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA,0.0,NaN,0.0%
2,1005337691,2.0,karen_aguilar@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,BUCARAMANGA,BUCARAMANGA,0.0,0.0,13.33%
3,1006053851,23.0,valentina_montoyac@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,CALI,CALI,2990000.0,130000.0,153.33%
4,1023938622,1.0,luis_munozmu@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,PEREIRA,PEREIRA,0.0,0.0,6.67%
5,1025521082,19.0,david_orjuelac@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,TUNJA,TUNJA,1900000.0,100000.0,126.67%
6,1033377195,27.0,laura_bedoyac@cun.edu.co,Gestor de ventas Junior,12.0,SANDRA,MEDELLIN,MEDELLIN,3240000.0,120000.0,225.0%
7,1036672351,0.0,juan_branpe@cun.edu.co,Gestor de ventas Junior,12.0,SANDRA,MEDELLIN,MEDELLIN,0.0,NaN,0.0%
8,1053616791,23.0,miryam_callejasp@cun.edu.co,Gestor de ventas Junior,15.0,SANDRA,TUNJA,TUNJA,2990000.0,130000.0,153.33%
9,1053787428,1.0,david_quiceno@cun.edu.co,Gestor de ventas Senior,8.0,SANDRA,MEDELLIN,MEDELLIN,0.0,0.0,12.5%


# SE ARMA LOS DATAFRAME FINAL

In [61]:
ventas_asesor_virtuales = ventas_asesor_virtuales.rename(columns={'Meta':'META_VIRTUAL'})
ventas_asesor_presenciales = ventas_asesor_presenciales.rename(columns={'Meta':'META_PRESENCIAL'})
ventas_asesor_especializaciones = ventas_asesor_especializaciones.rename(columns={'Meta':'META_ESPECIALIZACION'})
base_pago = ventas_asesor_virtuales.merge(
    ventas_asesor_presenciales[
        [
            'Documento Ejecutivo',
            'VENTAS_PRESENCIALES',
            'META_PRESENCIAL',
            'COMISION_TOTAL_PRESENCIAL',
            'COMISION_UNITARIA_PRESENCIAL',
            '%CUMPLIMIENTO_PRESENCIAL'
        ]
    ],
    how='left',
    on='Documento Ejecutivo'
).copy()


In [62]:
base_pago_final = base_pago.merge(
    ventas_asesor_especializaciones[
        [
       'Documento Ejecutivo', 'VENTAS_ESPECIALIZACION','META_ESPECIALIZACION', 
       'COMISION_TOTAL_ESPECIALIZACION', 'COMISION_UNITARIA_ESPECIALIZACION',
       '%CUMPLIMIENTO_ESPECIALIZACION'
        ]
    ],
    how='left',
    on='Documento Ejecutivo'
).copy()

In [63]:
columnas_ventas = [
    'VENTAS_VIRTUALES',
    'VENTAS_PRESENCIALES',
    'VENTAS_ESPECIALIZACION'
]

columnas_comisiones = [
    'COMISION_TOTAL_VIRTUAL',
    'COMISION_TOTAL_PRESENCIAL',
    'COMISION_TOTAL_ESPECIALIZACION'
]

base_pago_final['VENTAS_TOTALES'] = (
    base_pago_final[columnas_ventas].fillna(0).sum(axis=1)
)

base_pago_final['COMISION_FINAL'] = (
    base_pago_final[columnas_comisiones].fillna(0).sum(axis=1)
)
base_pago_final['PERIODO_PAGO'] = periodo_pago
base_pago_final = base_pago_final[['Documento Ejecutivo','GESTOR CORREO', 'CARGO', 'LIDER', 'SEDE', 'LOCALIDAD', 'PERIODO_PAGO','VENTAS_VIRTUALES', 
       'META_VIRTUAL','%CUMPLIMIENTO_VIRTUAL', 'COMISION_UNITARIA_VIRTUAL','COMISION_TOTAL_VIRTUAL',
       'VENTAS_PRESENCIALES', 'META_PRESENCIAL','%CUMPLIMIENTO_PRESENCIAL','COMISION_UNITARIA_PRESENCIAL','COMISION_TOTAL_PRESENCIAL',
       'VENTAS_ESPECIALIZACION', 'META_ESPECIALIZACION', '%CUMPLIMIENTO_ESPECIALIZACION', 'COMISION_UNITARIA_ESPECIALIZACION',
       'COMISION_TOTAL_ESPECIALIZACION','VENTAS_TOTALES', 'COMISION_FINAL']]

# SE ARMA EL ARCHIVO FINAL

In [64]:
df_historico = df_historico[df_historico['PERIODO_PAGO'].astype(str) == periodo_pago]
df_historico.shape

(1357, 19)

In [65]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
            select * from CUN_REPOSITORIO.AARBOLEDA.NOVALIDOS_VINCULACIONES_HIST
                         WHERE PERIODO_PAGO = ({periodo_pago})
            ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico_novalidos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_historico_novalidos.shape

(160, 16)

In [66]:
df_historico_novalidos_2 = df_historico_novalidos[~df_historico_novalidos['DOC_ALUM'].astype(str).isin(df_validos_virtuales_final['DOC_ALUM'])].copy()
df_historico_novalidos_2.shape

(116, 16)

In [67]:
df_novalidos_virtual_final = df_novalidos_virtual_final.drop(columns=['Llave'])
no_validos_total = pd.concat([df_historico_novalidos_2,df_novalidos_virtual_final])
no_validos_total['PERIODO_PAGO'] = periodo_pago
no_validos_total = no_validos_total[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo','Documento Ejecutivo', 'Regional',
       'PERIODO_PAGO', 'MOTIVO']]
no_validos_total.shape

(229, 16)

In [68]:
df_validos_virtuales_final['PERIODO_PAGO'] = periodo_pago
df_validos_virtuales_final = df_validos_virtuales_final[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo','Propietario_de_Oportunidad_Name', 'Fase', 'Hora_de_modificación', 'PERIODO_PAGO']]

In [70]:
with pd.ExcelWriter(
    f"LIQUIDACION_VINCULACIONES_REGIONALES_{periodo_pago}_2.xlsx",
    engine="openpyxl"
) as writer:

    base_pago_final.to_excel(
        writer,
        sheet_name="Comisiones_Asesores",
        index=False
    )

    df_historico.to_excel(
        writer,
        sheet_name="Validos",
        index=False
    )
    df_validos_virtuales_final.to_excel(
        writer,
        sheet_name="Novedades_Validas",
        index=False
    )

    no_validos_total.to_excel(
        writer,
        sheet_name="No_Validas",
        index=False
    )